# Lección 14.4 — Gradientes de política y PPO: más de lo que funcionó

> **Módulo 14 · Aprendizaje por refuerzo (introducción conceptual)** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lecciones 14.1–14.3; Lección 3.3 (descenso de gradiente) y Lección 3.4 (tasa de aprendizaje)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar la diferencia entre aprender **valores** (Q-learning) y aprender **directamente la política** (gradientes de política).
- Describir con palabras cómo aprende **REINFORCE**: subir la probabilidad de lo que salió mejor de lo esperado.
- Explicar para qué sirven la **línea base** (*baseline*) y la **ventaja** (*advantage*).
- Entender la idea central de **PPO**: no dejar que la política cambie demasiado de golpe (recorte o *clipping*), y conocer sus hiperparámetros.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4)
def softmax(preferencias):
    e = np.exp(preferencias - preferencias.max())
    return e / e.sum()
print("✅ Todo listo")

---
## 1. 📖 La intuición: el tiro libre 🏀

Hay dos formas de aprender a encestar tiros libres:

- **El analista** (Q-learning, Lección 14.3): para cada posición del cuerpo y cada movimiento posible, estima "qué tan bueno es" y luego elige el mejor. Necesita una chuleta enorme.
- **El deportista** (gradientes de política): no calcula valores. Simplemente **repite el movimiento**, y cada vez que encesta, refuerza un poco ese gesto; cuando falla, lo corrige un poco. Ajusta **el hábito** directamente.

```
 Q-learning:            estado ─► valor de cada acción ─► elijo la mejor
 Gradiente de política: estado ─► probabilidad de cada acción ─► sorteo y ajusto las probabilidades
```

La regla del deportista cabe en una frase: **"más de lo que salió mejor de lo esperado, menos de lo que salió peor"**.

### ✋ Ejercicio 1 — Calentamiento
Un robot debe aprender a girar un volante **cualquier ángulo** entre −90° y +90° (no solo "izquierda" o "derecha"). ¿Por qué sería incómodo usar una tabla Q como la de la Lección 14.3?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Porque hay **infinitas acciones** posibles (23.7°, 23.71°, …). La tabla necesitaría una columna por ángulo y, para decidir, habría que buscar el máximo entre infinitas columnas. Un gradiente de política, en cambio, puede aprender directamente "gira alrededor de 25°, con un margen de ±3°". Por eso los gradientes de política dominan en **robótica** y en cualquier problema con **acciones continuas**.
</details>

---
## 2. 📖 El concepto: una política con probabilidades

En los gradientes de política, el agente tiene una **preferencia** (un número) por cada acción, y la convierte en **probabilidades** con la función *softmax*, la misma que usa una red neuronal para clasificar (Lección 9.3):

```
preferencias:   [ 0.0 , 0.0 , 2.0 ]
                   │ softmax │
probabilidades: [ 0.11, 0.11, 0.78 ]   ← siempre suman 1
```

Luego **sortea** una acción con esas probabilidades. Así la exploración viene "incluida": mientras ninguna probabilidad sea 0, todas las acciones se siguen probando de vez en cuando.

### 🔮 Ejercicio 2 — De preferencias a probabilidades
¿Qué probabilidades darán estas preferencias? Apuesta antes de ejecutar:
- a) `[0, 0, 0]`
- b) `[5, 5, 5]`
- c) `[0, 0, 10]`

✍️ *Mis predicciones:*

In [ ]:
for pref in [[0, 0, 0], [5, 5, 5], [0, 0, 2], [0, 0, 10]]:     # ✏️ prueba las tuyas
    print(f"preferencias {str(pref):<12} → probabilidades {softmax(np.array(pref, float)).round(3)}")

<details><summary><b>👀 Ver respuesta</b></summary>

- a) y b) dan lo mismo: **[0.333, 0.333, 0.333]**. Lo que importa son las **diferencias** entre preferencias, no su valor absoluto.
- c) **[0.000, 0.000, 1.000]** (redondeado): una diferencia de 10 vuelve la política casi **determinista**. Ya no explora.

Aprender una política = **ajustar las preferencias** para que las acciones buenas ganen probabilidad.
</details>

---
## 3. 📖 Cómo aprende: REINFORCE

El algoritmo más sencillo de gradiente de política se llama **REINFORCE** (Williams, 1992). Tras cada intento:

```
ventaja = recompensa obtenida − recompensa que esperaba (línea base)

si ventaja > 0  → subo la preferencia de la acción que hice   ("¡más de eso!")
si ventaja < 0  → la bajo                                     ("menos de eso")
tamaño del ajuste ∝ tasa de aprendizaje × ventaja
```

Es **descenso de gradiente** (Lección 3.3) pero **cuesta arriba**: en vez de bajar una pérdida, **sube** la recompensa esperada. De ahí el nombre "gradiente de política".

### 📖 Nuestro caso: el estilo de un chatbot 🤖💬
Un chatbot puede responder en **3 estilos**. Los usuarios califican cada respuesta del 0 al 10. En promedio (y el chatbot **no** lo sabe):

| Estilo | Calificación media |
|---|---|
| 0 · Corta | 6 |
| 1 · Detallada | 7 |
| 2 · Con ejemplos | **8** |

Cada calificación tiene bastante ruido (± 2 puntos): a veces una respuesta corta recibe un 9. ¿Aprenderá el chatbot a preferir el estilo con ejemplos?

In [ ]:
MEDIAS = np.array([6.0, 7.0, 8.0])     # secreto: calificación media de cada estilo
RUIDO = 2.0
ESTILOS = ["corta", "detallada", "con ejemplos"]

def reinforce(n_respuestas=500, tasa=0.05, linea_base=True, semilla=0):
    rng = np.random.default_rng(semilla)
    pref = np.zeros(3)                    # preferencias: empiezan iguales
    esperado = 0.0                        # línea base: la calificación que "espero" en promedio
    historia = []
    for _ in range(n_respuestas):
        p = softmax(pref)
        a = rng.choice(3, p=p)                                  # sortea un estilo
        r = rng.normal(MEDIAS[a], RUIDO)                        # el usuario califica
        ventaja = r - esperado if linea_base else r             # ¿mejor o peor de lo esperado?
        pref += tasa * ventaja * (np.eye(3)[a] - p)             # ← la regla de REINFORCE
        esperado += 0.05 * (r - esperado)                       # actualiza lo que espera
        historia.append(softmax(pref))
    return np.array(historia)

### 🔮 Ejercicio 3 — Mira aprender al chatbot
Antes de ejecutar: tras 500 respuestas, ¿qué probabilidad tendrá el estilo "con ejemplos"? ¿Llegará a 100 %?

✍️ *Mi predicción:*

In [ ]:
historia = reinforce(semilla=0)       # ✏️ cambia la semilla para ver otro chatbot

for i, nombre in enumerate(ESTILOS):
    plt.plot(historia[:, i], label=nombre)
plt.xlabel("Respuestas"); plt.ylabel("Probabilidad de usar el estilo"); plt.legend(); plt.grid(alpha=.3)
plt.title("REINFORCE: el chatbot ajusta sus hábitos"); plt.show()
print("Probabilidades finales:", dict(zip(ESTILOS, historia[-1].round(3))))

<details><summary><b>👀 Ver respuesta</b></summary>

El estilo "con ejemplos" va ganando probabilidad poco a poco hasta superar el **90 %**, pero nunca llega exactamente a 100 %: siempre queda una pequeña probabilidad de probar los otros (exploración incluida).

Fíjate en que la curva es **ruidosa**: como las calificaciones tienen ruido, a veces una mala respuesta recibe buena nota y la probabilidad equivocada sube un poco. Es el precio de aprender solo con recompensas.
</details>

### 📖 ¿Para qué sirve la línea base?
Imagina que **no** restamos lo esperado (`linea_base=False`). Todas las calificaciones son positivas (6, 7, 8…), así que **cualquier** estilo que pruebe recibirá un "¡más de eso!". El chatbot solo aprende porque el estilo bueno recibe un empujón *un poco mayor*. Es como un profesor que felicita **todo**: sus alumnos tardan en saber qué hicieron bien.

Con la línea base, la señal es **relativa**: un 7 cuando esperabas 7.5 es una mala noticia. A esa diferencia se le llama **ventaja** (*advantage*): *¿qué tan mejor de lo normal fue esta acción?*

### 🔮 Ejercicio 4 — Con y sin línea base
Entrenaremos 200 chatbots de cada tipo. ¿Cuál terminará con mayor probabilidad promedio en el mejor estilo?

✍️ *Mi predicción:*

In [ ]:
for lb in [False, True]:
    finales = np.array([reinforce(tasa=0.05, linea_base=lb, semilla=s)[-1] for s in range(200)])
    print(f"línea base = {str(lb):<5} → prob. media del mejor estilo = {finales[:, 2].mean():.2f} · "
          f"chatbots atascados en un estilo peor = {(finales[:, :2].max(axis=1) > 0.9).mean():.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Sin línea base:** ≈ **0.78** de probabilidad media en el mejor estilo, y cerca de **1 de cada 5** chatbots se atasca en un estilo peor.
- **Con línea base:** ≈ **0.97**, y prácticamente ninguno se atasca.

Sin línea base, la primera respuesta (sea del estilo que sea) recibe un gran empujón solo por tener nota positiva, y eso puede encarrilar al chatbot en el estilo equivocado. **Restar lo esperado** hace que el aprendizaje sea mucho más estable. Todos los algoritmos modernos (incluido PPO) usan ventajas.
</details>

---
## 4. 📖 ¿Qué pasa si das pasos demasiado grandes?

En los gradientes de política hay un peligro especial. En aprendizaje supervisado, un paso demasiado grande se corrige en el siguiente, porque **los datos no cambian**. Aquí, en cambio, **la política decide qué datos verá después** (Lección 14.1):

```
paso enorme ─► la política casi deja de usar un estilo ─► ya no recibe datos de él
            ─► no puede descubrir que se equivocó ─► atascada para siempre 🔒
```

### 🔮 Ejercicio 5 — Tasa de aprendizaje gigante
Con línea base, compararemos `tasa` = 0.01, 0.05, 0.2 y 1.0. ¿Qué pasará con la tasa más grande?

✍️ *Mi predicción:*

In [ ]:
for tasa in [0.01, 0.05, 0.2, 1.0]:      # ✏️
    finales = np.array([reinforce(tasa=tasa, semilla=s)[-1] for s in range(200)])
    print(f"tasa = {tasa:<5} → prob. media del mejor = {finales[:, 2].mean():.2f} · "
          f"atascados en uno peor = {(finales[:, :2].max(axis=1) > 0.9).mean():.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **0.01:** demasiado lento: tras 500 respuestas aún no está seguro (≈ 0.81).
- **0.05:** el punto justo (≈ 0.97).
- **0.2:** empieza a fallar: 1 de cada 10 se atasca.
- **1.0:** **la mitad** de los chatbots se atasca en un estilo peor. Un par de calificaciones con suerte bastan para que la política se vuelva casi determinista… y ya no hay vuelta atrás.

Necesitamos una forma de dar pasos **grandes cuando es seguro** y **pequeños cuando no**. Esa es la idea de PPO.
</details>

### 📖 PPO: "mejora, pero no cambies demasiado"
**PPO** (*Proximal Policy Optimization*, OpenAI 2017) es hoy el algoritmo de RL más usado: entrena robots, agentes de videojuegos y fue el que se usó para el **RLHF** de los primeros ChatGPT (Lección 13.5).

Funciona por rondas:
1. Con la política actual (la "vieja"), recoge un **lote** de experiencias.
2. Reutiliza ese lote **varias veces** (épocas) para mejorar la política, pero vigilando el **cociente**:

```
cociente = probabilidad NUEVA de la acción / probabilidad VIEJA de la acción
```

3. Si el cociente se sale del rango **[1 − ε, 1 + ε]** (normalmente ε = 0.2, es decir, ±20 %), PPO **deja de empujar** en esa dirección. A esto se le llama **recorte** (*clipping*).

> 🧗 **Analogía:** subir una montaña con niebla atado a una cuerda de 20 metros. Puedes moverte con libertad, pero si te alejas más de la cuerda del último punto seguro, la cuerda te frena. Después de cada ronda, el punto seguro se mueve contigo.

### ✋ Ejercicio 6 — ¿Sigue empujando o ya no?
Con ε = 0.2, ¿PPO sigue empujando en estos casos? Recuerda: con ventaja positiva quiere **subir** la probabilidad; con negativa, **bajarla**.

| Caso | Ventaja | Cociente actual | ¿Sigue empujando? |
|---|---|---|---|
| A | +3 | 1.05 | ? |
| B | +3 | 1.35 | ? |
| C | −2 | 0.90 | ? |
| D | −2 | 0.70 | ? |

✍️ *Mis respuestas:*

In [ ]:
def ppo_sigue_empujando(ventaja, cociente, eps=0.2):
    if ventaja > 0 and cociente > 1 + eps: return "❌ no: ya subió más de un 20 %"
    if ventaja < 0 and cociente < 1 - eps: return "❌ no: ya bajó más de un 20 %"
    return "✅ sí"

for caso, v, c in [("A", 3, 1.05), ("B", 3, 1.35), ("C", -2, 0.90), ("D", -2, 0.70)]:
    print(f"Caso {caso}: ventaja {v:+}, cociente {c} → {ppo_sigue_empujando(v, c)}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **A:** sí (subió solo un 5 %).
- **B:** **no**: la acción ya es un 35 % más probable que en la política vieja; PPO no la sube más con este lote.
- **C:** sí (bajó solo un 10 %).
- **D:** **no**: ya bajó un 30 %.

Fíjate en la asimetría: si la ventaja es positiva y el cociente **bajó** (por ejemplo, 0.7), PPO **sí** sigue empujando para corregirlo. El recorte solo frena los cambios **en la dirección que el lote recomienda**, para que un lote con suerte no te lleve demasiado lejos.
</details>

▶️ Veámoslo en acción. Con el chatbot recién nacido (los 3 estilos al 33 %), recogemos **un lote de 20 respuestas** y lo reutilizamos **50 veces** para mejorar la política: sin recorte y con recorte.

In [ ]:
def recoger_lote(pref, n, rng):
    p = softmax(pref)
    a = rng.choice(3, size=n, p=p)
    r = rng.normal(MEDIAS[a], RUIDO)
    return a, r, p[a]                           # acciones, recompensas y probabilidad VIEJA de cada acción

def mejorar_con_lote(pref, a, r, p_vieja, epocas=50, tasa=0.1, eps=None):
    """Varias pasadas sobre el mismo lote. eps=None: sin recorte. eps=0.2: PPO."""
    pref = pref.copy()
    ventaja = r - r.mean()                      # línea base: la media del lote
    for _ in range(epocas):
        p = softmax(pref)
        cociente = p[a] / p_vieja
        empuja = np.ones(len(a), bool)
        if eps is not None:                     # el recorte de PPO
            empuja = ~(((ventaja > 0) & (cociente > 1 + eps)) | ((ventaja < 0) & (cociente < 1 - eps)))
        direccion = np.eye(3)[a] - p            # hacia dónde mover las preferencias para cada acción
        pref += tasa * ((empuja * cociente * ventaja)[:, None] * direccion).mean(axis=0)
    return pref

SEMILLA = 3     # ✏️ prueba 0, 1, 2, 3, 4
rng = np.random.default_rng(SEMILLA)
a, r, p_vieja = recoger_lote(np.zeros(3), 20, rng)
print("Nota media de cada estilo EN ESTE LOTE:", [round(r[a == i].mean(), 1) for i in range(3)])
print("Sin recorte →", softmax(mejorar_con_lote(np.zeros(3), a, r, p_vieja, eps=None)).round(2))
print("PPO (ε=0.2) →", softmax(mejorar_con_lote(np.zeros(3), a, r, p_vieja, eps=0.2)).round(2))

### 🔮 Ejercicio 7 — Un lote engañoso
Con `SEMILLA = 3`, mira las notas medias del lote: por azar, el estilo "detallada" sacó casi lo mismo que "con ejemplos". Antes de mirar las probabilidades: ¿cuál de las dos versiones crees que se dejará engañar más?

✍️ *Mi predicción:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **Sin recorte**, la política salta de 33 % a **60 %** en el estilo "detallada" (¡el que no es el mejor!) con un solo lote de 20 respuestas ruidosas.
- **Con PPO**, la política se mueve con prudencia (≈ 40 % / 40 %): mejora en la dirección que sugiere el lote, pero **no apuesta todo** a 20 datos.

Prueba otras semillas: cuando el lote es "honesto" (por ejemplo, la 1), sin recorte salta a **95 %** en el mejor estilo de una vez y PPO solo a **47 %**. PPO es **más lento por ronda**, pero mucho más difícil de engañar. Y como recoge un lote nuevo en cada ronda, avanza de forma segura.
</details>

In [ ]:
def entrenar(rondas=30, eps=None, tasa=0.3, semilla=0):
    rng = np.random.default_rng(semilla)
    pref = np.zeros(3)
    for _ in range(rondas):
        a, r, p_vieja = recoger_lote(pref, 20, rng)
        pref = mejorar_con_lote(pref, a, r, p_vieja, tasa=tasa, eps=eps)
    return softmax(pref)

for eps in [None, 0.2]:
    finales = np.array([entrenar(eps=eps, semilla=s) for s in range(200)])
    print(f"{'Sin recorte' if eps is None else 'PPO ε=0.2  '} → prob. media del mejor = {finales[:, 2].mean():.3f} · "
          f"atascados en uno peor = {(finales[:, :2].max(axis=1) > 0.9).mean():.1%}")

📝 Entrenamos 200 chatbots durante 30 rondas, con pasos algo más grandes (`tasa=0.3`). Ambos aprenden, pero **sin recorte** cerca de **1 de cada 6** chatbots queda atascado en un estilo peor; con PPO, ninguno. ✏️ Prueba `tasa=0.1` en `entrenar`: la diferencia se achica, pero no desaparece. En problemas grandes (robots, juegos, LLMs) esa diferencia es la que separa un entrenamiento que funciona de uno que se derrumba a mitad de camino.

### 📖 La pieza que falta: actor y crítico
En problemas con muchos pasos, ¿de dónde sale "lo que esperaba"? PPO entrena **dos** modelos a la vez:
- el **actor**: la política (qué hacer);
- el **crítico**: estima cuánto vale cada estado (como una tabla Q simplificada) y aporta la **línea base** para calcular la ventaja.

```
           ┌──────────┐   acción   ┌──────────┐
 estado ──►│  ACTOR   │───────────►│ ENTORNO  │──► recompensa
     │     └────▲─────┘            └──────────┘        │
     │          │ ventaja = "¿fue mejor de lo esperado?" │
     │     ┌────┴─────┐                                 │
     └────►│ CRÍTICO  │◄────────────────────────────────┘
           └──────────┘
```

Es la unión de las dos familias: **gradiente de política** (actor) + **valores** (crítico, la idea de la Lección 14.3).

### 📖 Para Colab: PPO de verdad
Con `stable-baselines3` entrenar PPO es casi una línea (requiere instalarlo; no corre aquí):

```python
# ⚠️ Para Colab
!pip install stable-baselines3 gymnasium
import gymnasium as gym
from stable_baselines3 import PPO

entorno = gym.make("CartPole-v1")
agente = PPO("MlpPolicy", entorno,
             learning_rate=3e-4,     # tasa de aprendizaje
             n_steps=2048,           # tamaño del lote de experiencias por ronda
             n_epochs=10,            # cuántas veces se reutiliza cada lote
             clip_range=0.2,         # el ε del recorte
             gamma=0.99,
             gae_lambda=0.95,        # cómo se calcula la ventaja en muchos pasos
             ent_coef=0.0,           # bono por mantener la política "abierta" (explorar)
             verbose=1)
agente.learn(total_timesteps=100_000)
```

---
## 5. 📖 Hiperparámetros de PPO

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `learning_rate` | Tamaño de cada paso de la red | Rápido pero inestable | Estable pero lento | 3·10⁻⁴ |
| `clip_range` (ε) | Cuánto puede cambiar la política por ronda | Cambios más audaces (menos protección) | Cambios más tímidos (más lento) | 0.2 (0.1–0.3) |
| `n_epochs` | Veces que se reutiliza cada lote | Aprovecha más cada lote, riesgo de sobreajustarse a él | Desperdicia datos | 10 (3–30) |
| `n_steps` / tamaño de lote | Experiencias por ronda | Ventajas menos ruidosas, rondas más lentas | Más rondas, más ruido | 2048 |
| `gamma` (γ) | Horizonte (Lección 14.2) | Más paciente | Más miope | 0.99 |
| `gae_lambda` (λ) | Equilibrio al estimar la ventaja: crítico vs. recompensas reales | Menos sesgo, más ruido | Menos ruido, más sesgo | 0.95 |
| `ent_coef` | Premio por mantener la política aleatoria | Explora más, tarda en decidirse | Se vuelve determinista antes | 0.0–0.01 |

### ✏️ Ejercicio 8 — La perilla del recorte
Volvamos al lote engañoso (`SEMILLA = 3`). Prueba `EPS` = **0.05**, **0.2** y **0.5**. ¿Qué tan lejos se mueve la política en cada caso?

✍️ *Mis observaciones:*

In [ ]:
SEMILLA = 3
EPS = 0.2       # ✏️ prueba 0.05, 0.2 y 0.5

rng = np.random.default_rng(SEMILLA)
a, r, p_vieja = recoger_lote(np.zeros(3), 20, rng)
print(f"ε = {EPS} →", softmax(mejorar_con_lote(np.zeros(3), a, r, p_vieja, eps=EPS)).round(2))

<details><summary><b>👀 Ver respuesta</b></summary>

Con estos datos:
- **ε = 0.05:** apenas se mueve (≈ 0.30 / 0.35 / 0.35): muy prudente; necesitará muchas rondas.
- **ε = 0.2:** ≈ 0.20 / 0.40 / 0.40: avanza sin comprometerse.
- **ε = 0.5:** ≈ 0.13 / 0.49 / 0.38: ya se parece a la versión sin recorte y empieza a creerse el engaño del lote.

`clip_range` es el "largo de la cuerda" de la analogía: más corta = más seguro y más lento.
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| La recompensa sube y de pronto **se desploma** | Pasos demasiado grandes: la política "saltó" a una zona mala | Bajar `learning_rate`, bajar `clip_range` o `n_epochs` |
| La política se vuelve determinista muy pronto y se estanca | Se acabó la exploración | Subir `ent_coef` |
| El aprendizaje es lentísimo y ruidoso | Ventajas muy ruidosas: lotes pequeños o sin línea base | Lotes más grandes, revisar el crítico, normalizar ventajas |
| Resultados muy distintos con otra semilla | El RL es muy sensible al azar | Repetir con 3–5 semillas y reportar media y dispersión |
| El agente encuentra un truco raro para subir la recompensa | *Reward hacking* (Lección 14.1) | Revisar la recompensa; en RLHF, penalizar alejarse del modelo original |

### 🔮 Ejercicio 9 — Diagnóstico
Un equipo entrena un robot con PPO. La recompensa media sube durante 2 horas y de repente cae a casi cero y no se recupera. En los registros ven que el cociente de probabilidades llegó a valores de **5** y **0.1** en algunas acciones. ¿Qué hiperparámetros revisarías primero?

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un cociente de 5 o 0.1 significa que la política cambió **muchísimo** en una sola ronda: justo lo que PPO intenta evitar. Revisaría `learning_rate` (bajarlo), `n_epochs` (menos pasadas por lote) y `clip_range` (asegurarse de que es ~0.2). Muchas implementaciones también cortan la ronda antes de tiempo si la política se aleja demasiado (*early stopping* por divergencia KL).
</details>

---
## 7. 📋 Ficha del modelo

| | REINFORCE | PPO |
|---|---|---|
| **Intuición** | "Más de lo que salió mejor de lo esperado" | Lo mismo, pero "sin alejarte demasiado de lo que ya funcionaba" |
| **Qué aprende** | La política directamente (probabilidades) | Política (actor) + valores (crítico) |
| **Cómo aprende** | Sube la probabilidad de cada acción según su ventaja | Reutiliza cada lote varias veces con un cociente recortado a [1 − ε, 1 + ε] |
| **Hiperparámetros clave** | Tasa de aprendizaje, línea base | `learning_rate`, `clip_range`, `n_epochs`, `n_steps`, γ, `gae_lambda`, `ent_coef` |
| **✅ Fortalezas** | Simple · Acciones continuas · Exploración incluida | Estable · Robusto · Funciona "de fábrica" en muchísimos problemas |
| **❌ Debilidades** | Muy ruidoso e inestable | Necesita mucha experiencia (*sample inefficient*) · Muchos hiperparámetros |
| **Úsalo cuando…** | Quieres entender la idea | Robótica, juegos, control continuo, RLHF |
| **Alternativas** | — | SAC (acciones continuas), DQN (pocas acciones discretas, Lección 14.3), DPO (alineación de LLMs sin RL explícito) |

## 8. 🧠 Resumen en 5 puntos
1. Los **gradientes de política** aprenden la política (probabilidades) **directamente**, sin pasar por una tabla de valores.
2. **REINFORCE**: sube la probabilidad de las acciones que salieron **mejor de lo esperado** y baja las demás.
3. Restar una **línea base** (usar la **ventaja**) reduce muchísimo el ruido y los atascos.
4. Pasos demasiado grandes pueden **atascar** la política, porque ella misma decide qué datos verá después.
5. **PPO** reutiliza cada lote varias veces, pero **recorta** los cambios a ±ε (≈ 20 %): es el estándar en robótica, juegos y RLHF.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** ¿Qué aprende directamente un método de gradiente de política?
- a) Una tabla Q · b) Las probabilidades de elegir cada acción · c) El entorno · d) La recompensa

**2.** Un agente recibió recompensa 7, y su línea base era 9. ¿Qué hace REINFORCE con la acción que eligió?
- a) Sube su probabilidad porque 7 es positivo · b) Baja su probabilidad porque salió peor de lo esperado · c) Nada · d) La elimina

**3.** En PPO con ε = 0.2, una acción con ventaja positiva ya es un 30 % más probable que en la política vieja. ¿Qué pasa?
- a) PPO la sigue subiendo · b) PPO deja de subirla con este lote · c) PPO la baja · d) Se reinicia el entrenamiento

**4.** ¿Cuál es el papel del **crítico** en PPO?
- a) Elegir la acción · b) Estimar cuánto vale cada estado para calcular la ventaja · c) Dar la recompensa · d) Recortar el cociente

**5.** ¿Para qué problema elegirías gradientes de política antes que una tabla Q?
- a) Un laberinto de 5×5 · b) Un brazo robótico que mueve 7 articulaciones con ángulos continuos · c) Elegir entre 3 anuncios · d) Tres en raya

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)**
2. **b)** La ventaja es 7 − 9 = −2.
3. **b)** Ya se salió del rango [0.8, 1.2] en la dirección que pide la ventaja.
4. **b)**
5. **b)** Acciones continuas y muchísimos estados.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** En el RLHF de un LLM (Lección 13.5), el "entorno" es una persona (o un modelo de recompensa) que califica respuestas, y la "política" es el propio LLM. ¿Por qué crees que ahí es especialmente importante que la política **no cambie demasiado** en cada ronda?

**B.** Tu compañero dice: *"PPO es mejor que REINFORCE, así que siempre aprende más rápido"*. ¿Estás de acuerdo? Usa lo que viste en el Ejercicio 7.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Porque el LLM ya sabe hablar bien gracias al pre-entrenamiento. Si la política cambia demasiado persiguiendo la recompensa, puede **romper** lo que ya sabía (empezar a escribir de forma rara) o encontrar trucos para engañar al modelo de recompensa (*reward hacking*: respuestas larguísimas, halagos, etc.). Por eso en RLHF, además del recorte de PPO, se añade una penalización por alejarse del modelo original (penalización KL).

**B.** No exactamente. **Por ronda**, PPO suele avanzar **menos** (en el Ejercicio 7, con un lote honesto, sin recorte saltaba a ~95 % y PPO solo a ~47 %). Su ventaja es la **estabilidad**: casi nunca se atasca ni se derrumba, así que **en el conjunto del entrenamiento** suele llegar más lejos y de forma más fiable.
</details>

### 9.3 🧩 Práctica integradora: "El ratón con hábitos"

Volvemos al pasillo de la Lección 14.1 (😾 en la casilla 0 con −10, 🧀 en la 6 con +10, −1 por paso, el ratón empieza en la 3). Esta vez el ratón **no** tendrá tabla Q: tendrá una **probabilidad de ir a la derecha en cada casilla** y la ajustará con REINFORCE.

Lo harás en 5 pasos cortos. **No avances sin revisar cada respuesta.**

**Paso 1 ▶️ — El ratón y su política**

In [ ]:
def prob_derecha(pref, casilla):
    return 1 / (1 + np.exp(-pref[casilla]))       # sigmoide (Lección 5.2): preferencia → probabilidad

def jugar(pref, rng, inicio=3, largo=7, max_pasos=50):
    casilla, casillas, acciones, recompensas = inicio, [], [], []
    for _ in range(max_pasos):
        derecha = int(rng.random() < prob_derecha(pref, casilla))
        casillas.append(casilla); acciones.append(derecha)
        casilla += 1 if derecha else -1
        if casilla == largo - 1: recompensas.append(10.0); break
        if casilla == 0:         recompensas.append(-10.0); break
        recompensas.append(-1.0)
    return casillas, acciones, recompensas

def reinforce_pasillo(episodios=300, tasa=0.05, semilla=0):
    rng = np.random.default_rng(semilla)
    pref, esperado, retornos = np.zeros(7), 0.0, []
    for _ in range(episodios):
        casillas, acciones, recompensas = jugar(pref, rng)
        lo_que_falta = np.cumsum(recompensas[::-1])[::-1]       # retorno desde cada paso hasta el final
        for c, a, G in zip(casillas, acciones, lo_que_falta):
            pref[c] += tasa * (G - esperado) * (a - prob_derecha(pref, c))
        esperado += 0.05 * (lo_que_falta[0] - esperado)
        retornos.append(sum(recompensas))
    return pref, np.array(retornos)

pref, retornos = reinforce_pasillo(semilla=0)
print(f"Retorno medio: primeros 50 episodios = {retornos[:50].mean():.1f} · últimos 50 = {retornos[-50:].mean():.1f}")

**Paso 2 🔮 — ¿Cuál es el máximo posible?** Recuerda la Lección 14.1: ¿qué retorno obtiene el ratón perfecto? ¿Qué tan cerca llegó el nuestro en los últimos 50 episodios? ¿Por qué no llega exactamente?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

El ratón perfecto (siempre a la derecha) obtiene **8**. El nuestro promedia **8.0** en los últimos 50 episodios: ¡ya se comporta como el ratón perfecto! Empezó en ≈ 5.2 en los primeros 50. Su política sigue siendo **probabilística**, pero la probabilidad de ir a la derecha en su camino ya es casi 1 (lo verás en el Paso 3). Con otras semillas suele quedar un poco por debajo (≈ 7.4 de media, Paso 4).
</details>

**Paso 3 🔮 — ¿Dónde aprendió más?** La celda siguiente muestra la probabilidad de ir a la derecha en cada casilla (1 a 5). Antes de ejecutar: ¿en qué casillas crees que el ratón estará **más seguro** y en cuáles **menos**? (Pista: ¿por dónde pasa más a menudo?)

✍️ *Mi predicción:*

In [ ]:
for c in range(1, 6):
    p = prob_derecha(pref, c)
    print(f"casilla {c}: P(➡️) = {p:.2f}  " + "█" * int(p * 30))

<details><summary><b>👀 Ver respuesta</b></summary>

Está **más seguro** en las casillas **3, 4 y 5** (por donde pasa en casi todos los episodios): 0.97 o más. En la **1** y sobre todo en la **2** está mucho menos seguro: ¡en la casilla 2 incluso prefiere ir a la **izquierda** (0.33)! Da igual mientras empiece en la 3, porque casi nunca pasa por ahí una vez que aprendió a ir a la derecha. Es la misma lección que con la tabla Q (Ejercicio 10 de la Lección 14.3): **un agente solo aprende bien lo que visita**.
</details>

**Paso 4 ✏️ — La tasa de aprendizaje.** Promediamos 50 ratones para cada tasa. Completa la lista con `0.005`, `0.05` y `0.2` y compara.

In [ ]:
TASAS = [0.005, 0.05, 0.2]     # ✏️

for tasa in TASAS:
    finales = [reinforce_pasillo(tasa=tasa, semilla=s)[1] for s in range(50)]
    finales = np.array(finales)
    print(f"tasa = {tasa:<6} → retorno medio: primeros 50 = {finales[:, :50].mean():5.2f} · últimos 50 = {finales[:, -50:].mean():5.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **0.005:** aprende lento (empieza ≈ −1.4 y acaba ≈ 6.6).
- **0.05:** ≈ 4.6 al principio y ≈ 7.4 al final: el mejor equilibrio.
- **0.2:** arranca algo más rápido (≈ 4.8) pero **termina peor** (≈ 6.4): sus pasos grandes lo hacen inestable, a veces "se pasa" y luego tiene que corregir.

La misma curva en "U" de siempre (Lecciones 3.4, 14.3 y la Sección 4 de esta lección).
</details>

**Paso 5 ✋ — Compara las dos familias.** Escribe en 3 frases cómo resolvería este mismo pasillo un agente de **Q-learning** (Lección 14.3) y en qué se diferencia de lo que hizo este ratón.

✍️ *Mi comparación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo:

> «El agente de Q-learning aprendería una **tabla** con el valor de ir a la izquierda y a la derecha en cada casilla, y luego elegiría siempre el mayor (más un poco de exploración con ε). Nuestro ratón, en cambio, nunca calculó cuánto vale cada acción: ajustó directamente **la probabilidad** de ir a la derecha, reforzando los movimientos de los episodios que salieron mejor de lo esperado. En un pasillo tan pequeño ambos funcionan; los gradientes de política brillan cuando las acciones son continuas o demasiadas para una tabla, y Q-learning cuando hay pocas acciones y queremos aprovechar experiencias viejas (*off-policy*).»
</details>

---
## 📚 Para profundizar
- 📘 **Libro:** Sutton y Barto, capítulo 13 (*Policy Gradient Methods*).
- 📰 **Artículo:** Schulman et al. (2017), *Proximal Policy Optimization Algorithms*: corto y muy citado.
- 📖 **Guía:** OpenAI, *Spinning Up in Deep RL*: explicaciones claras de REINFORCE, actor-crítico y PPO (inglés, gratis).
- 📺 **Video:** *The 37 Implementation Details of PPO* (blog de ICLR, 2022), para ver cuántos detalles importan en la práctica.

---
⬅️ **Anterior:** [14.3 Q-learning y redes Q profundas: la chuleta del agente](14.3-q-learning-y-dqn.ipynb) · ➡️ **Siguiente:** [14.5 Casos famosos: AlphaGo, robótica y RLHF](14.5-casos-famosos-alphago-robotica-rlhf.ipynb)